# Healthcare Contract Rates Comparison Report
**DF1**: Internal Team Extracted Data (`internal_team.csv`)  
**DF2**: Rules-Based Extracted Data (`input_data/contract_rates.xlsx`)  
Facility Filter: **Roswell Park Cancer Institute**

In [ ]:
import pandas as pd
import numpy as np
import re
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', 200)

## 1. Load Data

In [ ]:
# --- Load DF1: Internal Team CSV ---
df1_raw = pd.read_csv('internal_team.csv', dtype=str)
print(f"DF1 raw shape: {df1_raw.shape}")
df1_raw.head(3)

In [ ]:
# --- Load DF2: Rules-Based Excel ---
df2_raw = pd.read_excel('input_data/contract_rates.xlsx', dtype=str)
print(f"DF2 raw shape: {df2_raw.shape}")
df2_raw.head(3)

## 2. Prepare DF1

In [ ]:
# Columns of interest from DF1
DF1_COLS = {
    'Facility Name': 'facility_name',
    'Plan/Product':  'plan_product',
    'Service Name':  'service_name',
    'Rate':          'rate',
    'Mop':           'mop',
}

# Normalise column names (strip whitespace)
df1_raw.columns = df1_raw.columns.str.strip()

df1 = df1_raw[list(DF1_COLS.keys())].rename(columns=DF1_COLS).copy()

# Filter Roswell Park
df1 = df1[df1['facility_name'].str.contains('Roswell Park', case=False, na=False)].reset_index(drop=True)

# Normalise rate: strip currency symbols and commas, convert to float where possible
def normalise_rate(val):
    if pd.isna(val) or str(val).strip() == '':
        return np.nan
    cleaned = re.sub(r'[\$,]', '', str(val).strip())
    try:
        return float(cleaned)
    except ValueError:
        return cleaned  # keep text rates like 'By Report', 'Fee Schedule'

df1['rate'] = df1['rate'].apply(normalise_rate)

# Normalise string columns
for col in ['facility_name', 'plan_product', 'service_name', 'mop']:
    df1[col] = df1[col].astype(str).str.strip().str.lower()

df1['source'] = 'DF1 (Internal Team)'
print(f"DF1 (Roswell filtered): {df1.shape}")
df1.head()

## 3. Prepare DF2

In [ ]:
# Columns of interest from DF2
DF2_COLS = {
    'Facility Name': 'facility_name',
    'Plan/Product':  'plan_product',
    'Service Name':  'service_name',
    'Rate':          'rate',
    'Mop':           'mop',
}

df2_raw.columns = df2_raw.columns.str.strip()

df2 = df2_raw[list(DF2_COLS.keys())].rename(columns=DF2_COLS).copy()

# Filter Roswell Park
df2 = df2[df2['facility_name'].str.contains('Roswell Park', case=False, na=False)].reset_index(drop=True)

# Normalise rate: DF2 rate is string like '$5,000.00' or text
df2['rate'] = df2['rate'].apply(normalise_rate)

# Normalise string columns
for col in ['facility_name', 'plan_product', 'service_name', 'mop']:
    df2[col] = df2[col].astype(str).str.strip().str.lower()

df2['source'] = 'DF2 (Rules-Based)'
print(f"DF2 (Roswell filtered): {df2.shape}")
df2.head()

## 4. Comparison Logic
Match key: `service_name` + `plan_product` + `mop`  
Rate comparison: exact numeric match (tolerance ±0.01) or exact string match

In [ ]:
MATCH_KEYS = ['service_name', 'plan_product', 'mop']

def rate_match(r1, r2, tol=0.01):
    """Returns True if rates are considered equal."""
    if pd.isna(r1) and pd.isna(r2):
        return True
    if pd.isna(r1) or pd.isna(r2):
        return False
    if isinstance(r1, float) and isinstance(r2, float):
        return abs(r1 - r2) <= tol
    return str(r1).strip().lower() == str(r2).strip().lower()

# Merge on match keys (outer join)
merged = pd.merge(
    df1.drop(columns='source'),
    df2.drop(columns='source'),
    on=MATCH_KEYS,
    how='outer',
    suffixes=('_df1', '_df2'),
    indicator=True
)

# Classify rows
def classify(row):
    if row['_merge'] == 'both':
        if rate_match(row['rate_df1'], row['rate_df2']):
            return 'Matched'
        else:
            return 'Rate Mismatch'
    elif row['_merge'] == 'left_only':
        return 'In DF1 Only'
    else:
        return 'In DF2 Only'

merged['comparison_status'] = merged.apply(classify, axis=1)

# Summary counts
summary = merged['comparison_status'].value_counts().reset_index()
summary.columns = ['Status', 'Count']
summary['Percentage'] = (summary['Count'] / summary['Count'].sum() * 100).round(1).astype(str) + '%'
print("=== Comparison Summary ===")
print(summary.to_string(index=False))

## 5. Additional Analysis

In [ ]:
# --- MOP distribution ---
print("\n=== MOP Distribution — DF1 ===")
print(df1['mop'].value_counts().to_string())

print("\n=== MOP Distribution — DF2 ===")
print(df2['mop'].value_counts().to_string())

In [ ]:
# --- Rate statistics for numeric rows ---
df1_num = pd.to_numeric(df1['rate'], errors='coerce').dropna()
df2_num = pd.to_numeric(df2['rate'], errors='coerce').dropna()

rate_stats = pd.DataFrame({
    'Source': ['DF1 (Internal)', 'DF2 (Rules-Based)'],
    'Count': [len(df1_num), len(df2_num)],
    'Min': [df1_num.min(), df2_num.min()],
    'Max': [df1_num.max(), df2_num.max()],
    'Mean': [df1_num.mean(), df2_num.mean()],
    'Median': [df1_num.median(), df2_num.median()],
})
print("\n=== Rate Statistics (Numeric Rates Only) ===")
print(rate_stats.to_string(index=False))

In [ ]:
# --- Plan/Product coverage ---
plans_df1 = set(df1['plan_product'].dropna().unique())
plans_df2 = set(df2['plan_product'].dropna().unique())
print("\n=== Plan/Product Coverage ===")
print(f"Plans in DF1 only : {sorted(plans_df1 - plans_df2)}")
print(f"Plans in DF2 only : {sorted(plans_df2 - plans_df1)}")
print(f"Plans in both     : {sorted(plans_df1 & plans_df2)}")

In [ ]:
# --- Service name coverage ---
svc_df1 = set(df1['service_name'].dropna().unique())
svc_df2 = set(df2['service_name'].dropna().unique())
print("\n=== Service Name Coverage ===")
print(f"\nServices in DF1 only ({len(svc_df1 - svc_df2)}):")
for s in sorted(svc_df1 - svc_df2): print(f"  - {s}")
print(f"\nServices in DF2 only ({len(svc_df2 - svc_df1)}):")
for s in sorted(svc_df2 - svc_df1): print(f"  - {s}")
print(f"\nServices in both ({len(svc_df1 & svc_df2)}):")
for s in sorted(svc_df1 & svc_df2): print(f"  - {s}")

## 6. Export Excel Report

In [ ]:
OUTPUT_PATH = 'output_data/roswell_comparison_report.xlsx'
import os; os.makedirs('output_data', exist_ok=True)

# ---- Split merged into sub-dataframes ----
matched      = merged[merged['comparison_status'] == 'Matched']
rate_mismatch= merged[merged['comparison_status'] == 'Rate Mismatch']
df1_only     = merged[merged['comparison_status'] == 'In DF1 Only']
df2_only     = merged[merged['comparison_status'] == 'In DF2 Only']

DISPLAY_COLS = ['service_name', 'plan_product', 'mop',
                'rate_df1', 'rate_df2', 'facility_name_df1', 'facility_name_df2',
                'comparison_status']

def safe_cols(df, cols):
    return [c for c in cols if c in df.columns]

wb = Workbook()

HEADER_FILL   = PatternFill('solid', start_color='1F4E79')
HEADER_FONT   = Font(bold=True, color='FFFFFF', name='Arial', size=10)
MATCHED_FILL  = PatternFill('solid', start_color='C6EFCE')
MISMATCH_FILL = PatternFill('solid', start_color='FFEB9C')
DF1ONLY_FILL  = PatternFill('solid', start_color='BDD7EE')
DF2ONLY_FILL  = PatternFill('solid', start_color='FCE4D6')
SUMMARY_FILL  = PatternFill('solid', start_color='D9E1F2')
THIN = Side(style='thin', color='AAAAAA')
BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

def write_sheet(ws, df, title, row_fill=None, tab_color=None):
    ws.title = title
    if tab_color:
        ws.sheet_properties.tabColor = tab_color
    cols = safe_cols(df, DISPLAY_COLS)
    # Header
    for ci, col in enumerate(cols, 1):
        cell = ws.cell(row=1, column=ci, value=col.replace('_', ' ').title())
        cell.font = HEADER_FONT
        cell.fill = HEADER_FILL
        cell.alignment = Alignment(horizontal='center', wrap_text=True)
        cell.border = BORDER
        ws.column_dimensions[get_column_letter(ci)].width = max(15, len(col) + 4)
    # Rows
    for ri, (_, row) in enumerate(df[cols].iterrows(), 2):
        for ci, val in enumerate(row, 1):
            cell = ws.cell(row=ri, column=ci, value=val)
            cell.font = Font(name='Arial', size=9)
            cell.border = BORDER
            if row_fill:
                cell.fill = row_fill
    ws.freeze_panes = 'A2'

# Sheet 1: Summary
ws_sum = wb.active
ws_sum.title = 'Summary'
ws_sum.sheet_properties.tabColor = '1F4E79'

summary_data = [
    ['ROSWELL PARK CANCER INSTITUTE — CONTRACT RATES COMPARISON REPORT', ''],
    ['', ''],
    ['Metric', 'Value'],
    ['DF1 Total Rows (Roswell)',       len(df1)],
    ['DF2 Total Rows (Roswell)',        len(df2)],
    ['Matched (same key + rate)',       len(matched)],
    ['Rate Mismatch (same key)',        len(rate_mismatch)],
    ['In DF1 Only (not in DF2)',        len(df1_only)],
    ['In DF2 Only (not in DF1)',        len(df2_only)],
    ['Match Rate (%)',                  f"{round(len(matched)/max(len(merged),1)*100,1)}%"],
    ['', ''],
    ['Plan/Product Coverage', ''],
    ['Plans in DF1 only',              str(sorted(plans_df1 - plans_df2))],
    ['Plans in DF2 only',              str(sorted(plans_df2 - plans_df1))],
    ['Plans in both',                  str(sorted(plans_df1 & plans_df2))],
    ['', ''],
    ['Rate Statistics (Numeric)', ''],
    ['DF1 — Min Rate',                  df1_num.min() if len(df1_num) else 'N/A'],
    ['DF1 — Max Rate',                  df1_num.max() if len(df1_num) else 'N/A'],
    ['DF1 — Mean Rate',                 round(df1_num.mean(),2) if len(df1_num) else 'N/A'],
    ['DF2 — Min Rate',                  df2_num.min() if len(df2_num) else 'N/A'],
    ['DF2 — Max Rate',                  df2_num.max() if len(df2_num) else 'N/A'],
    ['DF2 — Mean Rate',                 round(df2_num.mean(),2) if len(df2_num) else 'N/A'],
]

for ri, row in enumerate(summary_data, 1):
    for ci, val in enumerate(row, 1):
        cell = ws_sum.cell(row=ri, column=ci, value=val)
        cell.font = Font(name='Arial', size=10, bold=(ri in [1,3,12,17]))
        cell.border = BORDER if ri >= 3 else Border()
        if ri == 1:
            cell.fill = PatternFill('solid', start_color='1F4E79')
            cell.font = Font(name='Arial', size=13, bold=True, color='FFFFFF')
        elif ri == 3:
            cell.fill = HEADER_FILL
            cell.font = Font(name='Arial', size=10, bold=True, color='FFFFFF')
        elif ri >= 4:
            cell.fill = SUMMARY_FILL

ws_sum.merge_cells('A1:B1')
ws_sum.column_dimensions['A'].width = 40
ws_sum.column_dimensions['B'].width = 60

# Sheet 2: Matched
ws2 = wb.create_sheet()
write_sheet(ws2, matched, 'Matched', row_fill=MATCHED_FILL, tab_color='00B050')

# Sheet 3: Rate Mismatch
ws3 = wb.create_sheet()
write_sheet(ws3, rate_mismatch, 'Rate Mismatch', row_fill=MISMATCH_FILL, tab_color='FF0000')

# Sheet 4: DF1 Only
ws4 = wb.create_sheet()
write_sheet(ws4, df1_only, 'In DF1 Only', row_fill=DF1ONLY_FILL, tab_color='0070C0')

# Sheet 5: DF2 Only
ws5 = wb.create_sheet()
write_sheet(ws5, df2_only, 'In DF2 Only', row_fill=DF2ONLY_FILL, tab_color='ED7D31')

# Sheet 6: Full Merged
ws6 = wb.create_sheet('Full Merged')
full_cols = safe_cols(merged, DISPLAY_COLS)
for ci, col in enumerate(full_cols, 1):
    cell = ws6.cell(row=1, column=ci, value=col.replace('_',' ').title())
    cell.font = HEADER_FONT; cell.fill = HEADER_FILL
    cell.alignment = Alignment(horizontal='center')
    ws6.column_dimensions[get_column_letter(ci)].width = 18

status_fills = {
    'Matched': MATCHED_FILL,
    'Rate Mismatch': MISMATCH_FILL,
    'In DF1 Only': DF1ONLY_FILL,
    'In DF2 Only': DF2ONLY_FILL,
}

for ri, (_, row) in enumerate(merged[full_cols].iterrows(), 2):
    status = merged.iloc[ri-2]['comparison_status']
    fill = status_fills.get(status, PatternFill())
    for ci, val in enumerate(row, 1):
        cell = ws6.cell(row=ri, column=ci, value=val)
        cell.font = Font(name='Arial', size=9)
        cell.fill = fill
        cell.border = BORDER
ws6.freeze_panes = 'A2'

wb.save(OUTPUT_PATH)
print(f"Report saved to: {OUTPUT_PATH}")

## 7. Quick Inline Preview

In [ ]:
print("\n=== MATCHED ROWS ===")
display(matched[safe_cols(matched, DISPLAY_COLS)].head(10))

print("\n=== RATE MISMATCH ROWS ===")
display(rate_mismatch[safe_cols(rate_mismatch, DISPLAY_COLS)].head(10))

print("\n=== IN DF1 ONLY ===")
display(df1_only[safe_cols(df1_only, DISPLAY_COLS)].head(10))

print("\n=== IN DF2 ONLY ===")
display(df2_only[safe_cols(df2_only, DISPLAY_COLS)].head(10))